# CEX–DEX arbitrage

Hourly pipeline: data → calibration → HJB → backtest.



In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import warnings
from pathlib import Path
import pickle

warnings.filterwarnings('ignore')

_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

plt.rcParams.update({'figure.dpi': 130, 'axes.titlesize': 11, 'axes.labelsize': 10})
%matplotlib inline
np.random.seed(42)

from arb.analysis import (
    bootstrap_ci_table,
    compute_descriptive_stats,
    compute_period_backtest,
    compute_q_regime_grid,
    compute_regime_comparison,
    compute_sensitivity_surface,
    detect_regimes,
    paired_ttest_strategies,
)
from arb.backtest import (
    backtest_summary,
    bootstrap_pnl,
    compute_sharpe,
    monte_carlo_eval,
    run_backtest,
    walk_forward_backtest,
)
from arb.calibrate import build_state_df, calibrate_all, calibrate_lambda_beta
from arb.config import *
from arb.models import build_A_t, build_K_t, g_closedform, mean_risk_objective
from arb.data import (
    binance_klines,
    binance_klines_paginated,
    binance_perp_klines,
    build_aligned,
    build_dex_1min_from_swaps,
    etherscan_gas_oracle,
    fetch_block_range_parallel,
    load_block_history,
    load_gecko_hourly,
    load_pool_hourly,
    load_pool_swaps,
)
from arb.hjb import solve_hjb_1d, solve_hjb_2d_xy
from arb.plots import (
    plot_2d_hjb_heatmap,
    plot_J_grid,
    plot_backtest,
    plot_correlation,
    plot_drift_check,
    plot_hjb_solution,
    plot_lambda_fit,
    plot_q_regime_heatmap,
    plot_regime_comparison,
    plot_returns_histogram_qq,
    plot_returns_rolling_vol,
    plot_sensitivity_surface_g_tau,
    plot_sensitivity_surface_sharpe,
    plot_2d_hjb_heatmap,
    savefig,
)

NB_OUT = Path(NB_ARTIFACT_DIR)
NB_OUT.mkdir(parents=True, exist_ok=True)
(NB_OUT / 'figures').mkdir(exist_ok=True)
print(NB_OUT)


## 1. Data


In [ ]:
cex_1h = binance_klines_paginated(
    CEX_SYMBOL, interval='1h', start_str='2021-01-01', use_cache=True
)
print(len(cex_1h), cex_1h.index[0].date(), cex_1h.index[-1].date())

cex_1d = binance_klines_paginated(
    CEX_SYMBOL, interval='1d', start_str='2020-01-01', use_cache=True
)
perp_1h = binance_perp_klines(PERP_SYMBOL, interval='1h', limit=1000)


In [ ]:
dex_1h = load_pool_hourly(POOL_ADDRESS, pages=60, use_cache=True)
swaps = load_pool_swaps(POOL_ADDRESS, pages=50, use_cache=True)
if dex_1h.empty:
    dex_1h = load_gecko_hourly(limit=1000, pages=10, use_cache=True)
assert not dex_1h.empty

px_1m = build_dex_1min_from_swaps(swaps)
if px_1m.empty:
    cache = DATA_DIR / 'hft_cache'
    for p in sorted(cache.glob('swaps_*.parquet')) if cache.exists() else []:
        sw = pd.read_parquet(p)
        raw = sw.set_index('dt')['dex_price'].sort_index()
        raw = raw[(raw > 100) & (raw < 1e6)]
        px_1m = raw.resample('1min').last().ffill(limit=5).rename('dex_mid').to_frame()
        if not px_1m.empty:
            break
print(len(dex_1h), len(swaps), len(px_1m))


In [ ]:
gas_oracle = etherscan_gas_oracle()
blocks = load_block_history(n_blocks=200, use_cache=True)

hist_specs = load_historical_blocks()
hist_blocks = {}
for name, spec in hist_specs.items():
    df = fetch_block_range_parallel(
        start_block=int(spec['start_block']),
        total_range=int(spec['total_range']),
        n_samples=int(spec.get('n_samples', 300)),
        use_cache=True,
    )
    hist_blocks[name] = df
    if not df.empty and 'baseFee_gwei' in df.columns:
        print(name, len(df), df['baseFee_gwei'].median())


In [ ]:
px = build_aligned(cex_1h, dex_1h, perp_1h if not perp_1h.empty else None)
print(len(px), px.index[0], px.index[-1])


In [ ]:
desc_stats = compute_descriptive_stats(px, blocks=blocks)
print(desc_stats)


## 2. Calibration


In [ ]:
assert not blocks.empty
params = calibrate_all(px, blocks, gas_oracle, swaps=swaps, Q=Q_ETH)
params['gamma'] = GAMMA
print(params['sigma_d'], params['sigma_c'], params['rho'], params['kappa'], params['beta'])
S0 = params['S0']


In [ ]:
state = build_state_df(px, params, Q=Q_ETH)
state = detect_regimes(state)
opp_base = state[state['is_opp']].copy()
print(len(state), len(opp_base))


## 3. Stylized facts


In [ ]:
assert not cex_1h.empty
fig = plot_drift_check(cex_1h); savefig(fig, 'drift'); plt.show()

fig = plot_returns_histogram_qq(px); savefig(fig, 'hist_qq'); plt.show()
fig = plot_returns_rolling_vol(px, roll_h=ROLL_H); savefig(fig, 'rolling_vol'); plt.show()

fig = plot_correlation(px, roll_h=ROLL_H)
savefig(fig, 'correlation'); plt.show()

rho_roll = px['ret_cex'].rolling(ROLL_H).corr(px['ret_dex'])
log_ret_h = np.log(cex_1h['close']).diff().dropna()
mu_h, sig_h = log_ret_h.mean(), log_ret_h.std()
tau_h = 1.0


## 4. HJB


In [ ]:
sig_d_h = params['sigma_d'] / np.sqrt(365 * 24)
K_ref = build_K_t(Q_ETH, S0, sig_d_h, params['rho'])
A_ref = build_A_t(Q_ETH, S0, sig_d_h, params['slip_alpha'], params['slip_beta'])
g_test = np.linspace(params['g_lo'], params['g_hi'], 500)
delta_ref = float(px['spread_usd'].abs().median())
J_grid = np.array([
    mean_risk_objective(
        g, K_ref, A_ref, S0, Q_ETH, delta_ref,
        params['kappa'], params['beta'], GAMMA,
    )
    for g in g_test
])
g_star_grid = g_test[np.argmax(J_grid)]
g_star_cf = g_closedform(
    K_ref, A_ref, S0, params['kappa'], params['beta'], GAMMA,
    g_lo=params['g_lo'], g_hi=params['g_hi'],
)
fig = plot_J_grid(g_test, J_grid, g_star_cf, g_star_grid)
savefig(fig, 'J_grid')
plt.show()


In [ ]:
u_hjb, phi_hjb, g_star_hjb = solve_hjb_1d(
    params, T_horizon=T_HORIZON, N_z=N_Z_HJB, N_t=N_T_HJB, gamma=GAMMA, Q=Q_ETH,
)
fig = plot_hjb_solution(u_hjb, phi_hjb, g_star_hjb, params=params)
savefig(fig, 'hjb_1d')
plt.show()


In [ ]:
cache = NB_OUT / 'cache'
cache.mkdir(exist_ok=True)
with open(cache / 'params.pkl', 'wb') as f:
    pickle.dump(params, f)
with open(cache / 'hjb_grids.pkl', 'wb') as f:
    pickle.dump({'u_grid': u_hjb, 'g_star': g_star_hjb, 'phi': phi_hjb}, f)


In [ ]:
if HJB_CONVERGENCE_CHECK:
    N_ref = min(N_Z_HJB_HEAVY, 4000)
    u2, phi2, g2 = solve_hjb_1d(params, N_z=N_ref, N_t=N_T_HJB_HEAVY,
                                  gamma=GAMMA, Q=Q_ETH)
    phi_ref_on_base = np.interp(u_hjb, u2, phi2)
    g_ref_on_base   = np.interp(u_hjb, u2, g2)
    max_phi_err = np.max(np.abs(phi_hjb - phi_ref_on_base))
    max_g_err   = np.max(np.abs(g_star_hjb - g_ref_on_base))
    print(max_phi_err, max_g_err)


## 5. Backtest


In [ ]:
dex_series = px_1m['dex_mid'] if not px_1m.empty else px['dex_mid']
def hjb_pol(row):
    u = np.log(max(float(row['cex_close']) / max(float(row.get('dex_mid', row['cex_close'])), 1e-6), 1e-6))
    return float(np.interp(np.clip(u, u_hjb[0], u_hjb[-1]), u_hjb, g_star_hjb))
def cf_pol(row):
    return g_closedform(
        row.get('K_t', K_ref), row.get('A_t', A_ref),
        float(row['cex_close']), params['kappa'], params['beta'], GAMMA,
        g_lo=params['g_lo'], g_hi=params['g_hi'],
    )
def naive_pol(row):
    return float(params.get(NAIVE_GAS_FIELD, params.get('propose_gwei', params['g_lo'])))
bt_dict = {
    'HJB': run_backtest(opp_base, hjb_pol, 'HJB', params, dex_series=dex_series),
    'CF': run_backtest(opp_base, cf_pol, 'CF', params, dex_series=dex_series),
    'Naive': run_backtest(opp_base, naive_pol, 'Naive', params, dex_series=dex_series),
}
fig = plot_backtest(bt_dict)
savefig(fig, 'backtest')
plt.show()
print(paired_ttest_strategies(bt_dict, reference='Naive'))
print(pd.DataFrame([backtest_summary(bt, n) for n, bt in bt_dict.items() if not bt.empty]).set_index('label'))
print(bootstrap_ci_table(bt_dict))


## 6. Gas regimes


In [ ]:
gas_regimes = {'calm_current': dict(params)}

for name, blk in hist_blocks.items():
    if blk.empty or 'baseFee_gwei' not in blk.columns:
        continue
    k, b, _ = calibrate_lambda_beta(blk['baseFee_gwei'].dropna().values)
    if k is None:
        continue
    p = dict(params)
    bf = blk['baseFee_gwei']
    p.update({
        'kappa': k,
        'beta': b,
        'base_fee': float(bf.median()),
        'propose_gwei': float(bf.quantile(0.7)),
        'g_lo': float(bf.median()) + 0.05,
        'g_hi': float(bf.quantile(0.95)) * 2.0,
    })
    gas_regimes[name] = p

for rname, synth in SYNTH_REGIMES.items():
    if rname in gas_regimes:
        continue
    p = dict(params)
    p.update(synth)
    p['g_lo'] = synth['base_fee'] + 0.05
    p['g_hi'] = synth['propose_gwei'] * 3.0
    gas_regimes[rname] = p


In [ ]:
gas_regimes = {'calm_current': dict(params)}

for name, blk in hist_blocks.items():
    if blk.empty or 'baseFee_gwei' not in blk.columns:
        continue
    k, b, _ = calibrate_lambda_beta(blk['baseFee_gwei'].dropna().values)
    if k is None:
        continue
    p = dict(params)
    bf = blk['baseFee_gwei']
    p.update({
        'kappa': k,
        'beta': b,
        'base_fee': float(bf.median()),
        'propose_gwei': float(bf.quantile(0.7)),
        'g_lo': float(bf.median()) + 0.05,
        'g_hi': float(bf.quantile(0.95)) * 2.0,
    })
    gas_regimes[name] = p

for rname, synth in SYNTH_REGIMES.items():
    if rname in gas_regimes:
        continue
    p = dict(params)
    p.update(synth)
    p['g_lo'] = synth['base_fee'] + 0.05
    p['g_hi'] = synth['propose_gwei'] * 3.0
    gas_regimes[rname] = p


In [ ]:
regime_comp = compute_regime_comparison(
    opp_base, params, gas_regimes, u_hjb, g_star_hjb, dex_series=dex_series, Q=Q_ETH,
)
print(regime_comp.round(3))
fig = plot_regime_comparison(regime_comp)
if fig:
    savefig(fig, 'regime_cmp')
    plt.show()


In [ ]:
gas_regimes = {'calm_current': dict(params)}

for name, blk in hist_blocks.items():
    if blk.empty or 'baseFee_gwei' not in blk.columns:
        continue
    k, b, _ = calibrate_lambda_beta(blk['baseFee_gwei'].dropna().values)
    if k is None:
        continue
    p = dict(params)
    bf = blk['baseFee_gwei']
    p.update({
        'kappa': k,
        'beta': b,
        'base_fee': float(bf.median()),
        'propose_gwei': float(bf.quantile(0.7)),
        'g_lo': float(bf.median()) + 0.05,
        'g_hi': float(bf.quantile(0.95)) * 2.0,
    })
    gas_regimes[name] = p

for rname, synth in SYNTH_REGIMES.items():
    if rname in gas_regimes:
        continue
    p = dict(params)
    p.update(synth)
    p['g_lo'] = synth['base_fee'] + 0.05
    p['g_hi'] = synth['propose_gwei'] * 3.0
    gas_regimes[rname] = p


## 6.2 Sensitivity


In [ ]:
gammas_s = np.array([1.5, 2.0, 3.0, 5.0, 8.0])
Qs_s = np.array([0.1, 0.5, 1.0, 2.0, 5.0, 10.0])
median_sp = float(px['spread_usd'].abs().median()) if not px.empty else 0.5

g_surf, g_raw_surf, tau_surf, sh_surf = compute_sensitivity_surface(
    params, gammas_s, Qs_s, median_spread_usd=median_sp, S=S0)

fig = plot_sensitivity_surface_g_tau(gammas_s, Qs_s, g_raw_surf, tau_surf)
savefig(fig, 'sens_g_tau'); plt.show()
fig = plot_sensitivity_surface_sharpe(gammas_s, Qs_s, sh_surf)
savefig(fig, 'sens_sharpe'); plt.show()


## 2D HJB


In [ ]:
if HJB_2D_ACTIVE:
    xg_2d, yg_2d, phi_2d, g_star_2d, meta_2d = solve_hjb_2d_xy(
        params, T_horizon=T_HORIZON,
        Nx=N_X_HJB_XY, Ny=N_Y_HJB_XY, N_t=N_T_HJB_XY,
        gamma=GAMMA, Q=Q_ETH, n_g=G_SCAN,
        )

    _hjb2d_cache = NB_OUT / 'figures' / 'hjb_2d_xy_cache.npz'
    np.savez_compressed(_hjb2d_cache, xg_2d=xg_2d, yg_2d=yg_2d, phi_2d=phi_2d, g_star_2d=g_star_2d)

    fig = plot_2d_hjb_heatmap(xg_2d, yg_2d, phi_2d, g_star_2d)
    savefig(fig, 'hjb_2d'); plt.show()

    mid_ix = len(xg_2d) // 2
    u_2d_slice    = xg_2d[mid_ix] - yg_2d
    phi_2d_slice  = phi_2d[:, mid_ix]
    g_2d_slice    = g_star_2d[:, mid_ix]

    fig = plot_hjb_solution(u_hjb, phi_hjb, g_star_hjb, params=params,
                            u_1d=u_2d_slice, phi_1d=phi_2d_slice, g_1d=g_2d_slice)
    savefig(fig, 'hjb_2d_vs_1d'); plt.show()


In [ ]:
cache_npz = NB_OUT / 'figures' / 'hjb_2d_xy_cache.npz'
z = np.load(cache_npz)
fig = plot_2d_hjb_heatmap(z['xg_2d'], z['yg_2d'], z['phi_2d'], z['g_star_2d'])
savefig(fig, 'hjb_2d')
plt.show()


## 5.2 Walk-forward


In [ ]:
wf_results = walk_forward_backtest(
    state, params, u_hjb, g_star_hjb,
    dex_series=px_1m['dex_mid'] if not px_1m.empty else px['dex_mid'],
    wf_train=WF_TRAIN, wf_test=WF_TEST,
)
print(len(wf_results))


## 5.2 Walk-forward


In [ ]:
PNL_COL = 'net_pnl'
valid = {k: v for k, v in bt_dict.items() if v is not None and not v.empty}

def risk_row(bt):
    p = bt[PNL_COL].values.astype(float)
    return {
        'n': len(p),
        'mean': p.mean(),
        'std': p.std(),
        'sharpe': p.mean() / p.std() if p.std() > 0 else 0.0,
        'VaR_95': np.percentile(p, 5),
    }

print(pd.DataFrame({n: risk_row(bt) for n, bt in valid.items()}).round(4))
naive_p = valid['Naive'][PNL_COL].values
for nm in ('HJB', 'CF'):
    if nm in valid:
        _, p = levene(valid[nm][PNL_COL].values, naive_p)
        print(nm, p)

fig, ax = plt.subplots(figsize=(10, 4))
for nm, bt in valid.items():
    bt['cum_pnl'].plot(ax=ax, label=nm)
ax.legend()
ax.set_ylabel('cum PnL')
savefig(fig, 'risk_profile')
plt.show()
